# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² tabular dataset (https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements (record sets, fields, columns) are referenced strictly by their `@id` fields to ensure clarity and completeness.

### Dataset Source

This notebook uses the Croissant schema at: 
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading

In this section, we load the dataset's metadata and prepare to access its record sets and fields via their `@id`s.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Fetch and display summary information from metadata
meta = dataset.metadata
print(f"{meta.name}\n")
print(meta.description)

## 2. Data Overview

Let's list the available record sets and their fields, referencing each by its `@id`.

We will use `dataset.record_sets` and print out `@id`, `name`, and fields for each set.

In [ ]:
# List all available record sets and enumerate their fields 
record_set_ids = []

print("\nAvailable record sets and fields (referenced by @id):\n------------------------------")
for rs in dataset.record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - {fld.name} (@id: {fld.id}, type: {fld.data_type})")
    print("")

if not record_set_ids:
    print("No record sets found. Please verify the schema or review dataset metadata.")

## 3. Data Extraction

Extract records from each record set using its `@id`. We'll load each into a pandas DataFrame for easier manipulation and analysis.

We reference record sets and fields strictly by their `@id`.

In [ ]:
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} records. Columns (@id):")
        print(f"    {df.columns.tolist()}\n")
    except Exception as e:
        print(f"  Could not load records for {rs_id}. Error: {e}\n")

# For illustration, print head of the first available record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"First rows of RecordSet @id: {first_rs_id}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll conduct basic inspection, normalization, and groupings over a numeric field.

- **You can select a specific RecordSet and a numeric field by their `@id` from the overview above.**
- We'll filter and normalize a numeric field and perform a groupby operation if a grouping field is available.

In [ ]:
# Example: Choose the main tabular record set and a numeric field (@id)
# Substitute these values with the actual @id from your overview, e.g.,
# main_rs_id = 'cr:RecordSet/second_primary_colorectal_cancer' or similar

main_rs_id = None
for rs in dataset.record_sets:
    # Heuristic: choose the first record set with >0 records and at least 1 numerical field
    df = dataframes.get(rs.id)
    if df is not None and len(df) > 0:
        for fld in rs.fields:
            if fld.data_type in ["Float", "Number", "Integer"] and fld.id in df.columns:
                main_rs_id = rs.id
                numeric_field_id = fld.id
                break
        if main_rs_id is not None:
            break

if main_rs_id is None:
    raise Exception("No suitable record set and numeric field found. Please check data import.")

df = dataframes[main_rs_id]
print(f"Using RecordSet: {main_rs_id}, Numeric field: {numeric_field_id}")

# Display value counts and statistics for the numeric field
print("\nBasic statistics for numeric field:")
print(df[numeric_field_id].describe())

# Set a threshold (for demonstration, e.g. the mean)
threshold = df[numeric_field_id].mean()

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the chosen numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping by a candidate categorical field
group_field_id = None
for fld in dataset.record_sets_by_id[main_rs_id].fields:
    if fld.data_type == "Text" and fld.id in filtered_df.columns:
        group_field_id = fld.id
        break

if group_field_id is not None:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped.head())
else:
    print("No suitable categorical group field found in filtered DataFrame.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship with a categorical field (if present).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the numeric field distribution after filtering
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered > mean)")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Optional: Plot boxplot grouped by group_field_id if available
if group_field_id is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

Using `mlcroissant`, we've:
- Loaded and explored the FAIR² dataset's metadata and record structure strictly by `@id` fields.
- Inspected tabular data, performed basic numeric and grouping operations, and visualized distributions.
- Demonstrated how to prepare the dataset for further analysis or ML applications, maintaining full traceability of all referenced data fields.

**Next steps:** You can extend this notebook with detailed hypothesis testing, advanced ML modeling, or merge with external ontologies using the Croissant schema's semantic capabilities.